In [16]:
import os
import re
import json
import hashlib
from datetime import date
from textwrap import dedent
from typing import List, Dict, Any, Optional
from neo4j import GraphDatabase
from dotenv import load_dotenv
import re, hashlib
from datetime import date
from typing import Dict, Any, List, Optional
load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "3261-PArk")
POLICY_URI = os.getenv("POLICY_URI", "blob://policies/aup_v1.pdf")



In [17]:
# --------- Helpers ---------
def sha256(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def compact_whitespace(s: str) -> str:
    return re.sub(r"[ \t]+", " ", s).strip()

def normalize_title(s: str) -> str:
    return compact_whitespace(s.replace("  ", " "))

def make_id(policy_id: str, number: str) -> str:
    safe = number.replace(" ", "").replace("•", "-")
    return f"{policy_id}:{safe}"

# --------- Parsing ---------
SECTION_RE   = re.compile(r"^(\d+)\.\s+([A-Z].+)$")
CLAUSE_RE    = re.compile(r"^(\d+\.\d+(?:\.\d+)*)\s+(.+)$")
LIST_HEAD_RE = re.compile(r"^(3\.7\.\d+)\.\s+(.+)$")
BULLET_RE    = re.compile(r"^[•\-]\s+(.*)$")

In [18]:
# Helpers (generic)
# -------------------------
def sha256(s: str) -> str:
    import hashlib
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def compact(s: str) -> str:
    return re.sub(r"[ \t]+", " ", s).strip()

def make_id(policy_id: str, number: str) -> str:
    safe = number.replace(" ", "").replace("•", "-")
    return f"{policy_id}:{safe}"

In [19]:
# Configurable patterns

# Section headings (choose what you need; first that matches wins)
SECTION_RES = [
    re.compile(r"^(\d+)\.\s+(.+)$"),            # "1. PURPOSE"
    re.compile(r"^([IVXLCDM]+)\.\s+(.+)$"),     # "I. Scope"
    re.compile(r"^([A-Z])\.\s+(.+)$"),          # "A. Overview"
]

# Numbered clauses like "3.1", "3.1.4", "2.3.1.1", with optional trailing dot then text
CLAUSE_RE = re.compile(r"^(\d+(?:\.\d+)+)\.?\s+(.*\S.*)$")

# A “list header” is a numbered clause followed by a Title/Heading (kept) and then bullets
LIST_HEADER_RE = re.compile(r"^(\d+(?:\.\d+)+)\.?\s+([A-Z].+)$")

# Bullets: "• text", "- text", "* text", "● text"
BULLET_RE = re.compile(r"^[\-\*\u2022\u25CF]\s+(.*\S.*)$")

# -------------------------
# Category mapping (optional)
# -------------------------
def infer_category(number: str, section_title: str, category_map: Optional[Dict[str,str]] = None) -> str:
    """
    Fallback rule:
    - If a prefix of the clause number is in category_map (e.g., '3.2' -> 'Internet'), use that.
    - Else use a normalized section title as category.
    """
    if category_map:
        # try longest prefix match on dotted numbers, e.g., '3.5.11.1' -> '3.5.11' -> '3.5' -> '3'
        parts = number.split('.')
        for k in range(len(parts), 0, -1):
            key = '.'.join(parts[:k])
            if key in category_map:
                return category_map[key]
    return re.sub(r"\W+", "", section_title).strip() or "General"

# -------------------------
# Core parser (framework-agnostic)
# -------------------------
def parse_policy(
    text: str,
    policy_id: str,
    title: str,
    version: str,
    *,
    owner: str = "Policy Owner",
    policy_uri: Optional[str] = None,
    category_map: Optional[Dict[str,str]] = None
) -> Dict[str, Any]:
    """
    Parse ANY numbered policy into sections and clauses.
    - Sections: matched by SECTION_RES (1., I., A., …)
    - Clauses:   matched by CLAUSE_RE (1.1, 1.2.3, …)
    - List headers: numbered clause with a TitleCase trailing text; bullets under it become individual clauses
    - Bullets: each bullet line becomes a clause "X.Y • N"

    Returns a dict with Policy + sections[*].clauses[*]
    """
    lines = [l.rstrip() for l in text.splitlines()]

    # 1) Find section start indices
    sec_starts: List[int] = []
    for i, ln in enumerate(lines):
        for rx in SECTION_RES:
            if rx.match(ln):
                sec_starts.append(i)
                break
    if not sec_starts:
        # no explicit sections detected; treat whole doc as section "1"
        lines = [f"1. {title}"] + lines
        sec_starts = [0]

    sec_starts.append(len(lines))

    sections: List[Dict[str, Any]] = []

    # 2) Walk sections
    for si in range(len(sec_starts)-1):
        s_start, s_end = sec_starts[si], sec_starts[si+1]
        head = lines[s_start]
        sec_num, sec_title = _match_section(head)
        sec_body = [l for l in lines[s_start+1:s_end]]
        sec_text = "\n".join(sec_body).strip()

        sec = {
            "id": f"{policy_id}:S{sec_num}",
            "number": sec_num,
            "title": compact(sec_title),
            "text": sec_text,
            "sha256": sha256(sec_text),
            "clauses": []
        }

        # 3) Clause extraction within this section
        current_list_number = None   # e.g., "3.7.2"
        current_list_title  = None
        bullet_index = 0

        i = 0
        while i < len(sec_body):
            raw = sec_body[i].strip()
            if not raw:
                i += 1
                continue

            # a) Numbered clause (generic)
            m = CLAUSE_RE.match(raw)
            if m:
                number, rest = m.group(1), m.group(2).strip()

                # Try to detect "list header": a numbered heading followed by bullets.
                mlh = LIST_HEADER_RE.match(raw)
                if mlh and _peek_has_bullet(sec_body, i+1):
                    current_list_number = mlh.group(1)          # e.g., "3.7.2"
                    current_list_title  = compact(mlh.group(2)) # e.g., "System and Network Activities"
                    bullet_index = 0
                    # Optionally persist the list header as a heading node
                    sec["clauses"].append({
                        "id": make_id(policy_id, number),
                        "number": number,
                        "title": current_list_title,
                        "text": "",
                        "category": infer_category(number, sec_title, category_map),
                        "section_number": sec_num,
                        "path": f"{title} > {sec_num} {sec_title} > {number} {current_list_title}",
                        "sha256": sha256(""),
                        "is_heading": True
                    })
                    i += 1
                    continue

                # Regular numbered clause
                sec["clauses"].append({
                    "id": make_id(policy_id, number),
                    "number": number,
                    "title": None,
                    "text": rest,
                    "category": infer_category(number, sec_title, category_map),
                    "section_number": sec_num,
                    "path": f"{title} > {sec_num} {sec_title} > {number}",
                    "sha256": sha256(rest),
                    "is_heading": False
                })
                i += 1
                continue

            # b) Bullet line under a prior list header
            b = BULLET_RE.match(raw)
            if b and current_list_number:
                bullet_index += 1
                number = f"{current_list_number} • {bullet_index}"
                text_ = compact(b.group(1))
                sec["clauses"].append({
                    "id": make_id(policy_id, number),
                    "number": number,
                    "title": current_list_title,
                    "text": text_,
                    "category": infer_category(current_list_number, sec_title, category_map),
                    "section_number": sec_num,
                    "path": f"{title} > {sec_num} {sec_title} > {current_list_number} {current_list_title} > • {bullet_index}",
                    "sha256": sha256(text_),
                    "is_heading": False
                })
                i += 1
                continue

            # c) If we hit a non-bullet after finishing a list, reset list state
            if current_list_number and not b:
                current_list_number = None
                current_list_title = None
                bullet_index = 0
                # fall through to treat this line again (don’t skip)
                # but avoid infinite loop
                # If it's a plain paragraph (no number/bullet), you can:
                # - attach as a "context" clause, OR
                # - ignore. Here we attach as context for traceability.
                if raw:
                    number = f"{sec_num}.context.{sha256(raw)[:6]}"
                    sec["clauses"].append({
                        "id": make_id(policy_id, number),
                        "number": number,
                        "title": f"{sec_title} Context",
                        "text": raw,
                        "category": infer_category(sec_num, sec_title, category_map),
                        "section_number": sec_num,
                        "path": f"{title} > {sec_num} {sec_title} > Context",
                        "sha256": sha256(raw),
                        "is_heading": False
                    })
                    i += 1
                continue

            # d) Plain paragraph (no number, no bullet)
            if raw:
                number = f"{sec_num}.context.{sha256(raw)[:6]}"
                sec["clauses"].append({
                    "id": make_id(policy_id, number),
                    "number": number,
                    "title": f"{sec_title} Context",
                    "text": raw,
                    "category": infer_category(sec_num, sec_title, category_map),
                    "section_number": sec_num,
                    "path": f"{title} > {sec_num} {sec_title} > Context",
                    "sha256": sha256(raw),
                    "is_heading": False
                })
            i += 1

        sections.append(sec)

    policy_doc = {
        "id": policy_id,
        "title": title,
        "version": version,
        "effective_on": date.today().isoformat(),
        "owner": owner,
        "uri": policy_uri,
        "sha256": sha256(text),
        "sections": sections
    }
    return policy_doc

# -------------------------
# Internals
# -------------------------
def _match_section(line: str):
    for rx in SECTION_RES:
        m = rx.match(line)
        if m:
            return m.group(1), m.group(2)
    # Fallback: treat first line as "1. <line>"
    return "1", line

def _peek_has_bullet(lines: List[str], start_idx: int, lookahead: int = 3) -> bool:
    """
    Return True if one of the next few non-empty lines is a bullet.
    Helps decide if a numbered line is a list header.
    """
    seen = 0
    for j in range(start_idx, min(len(lines), start_idx + 30)):
        l = lines[j].strip()
        if not l:
            continue
        if BULLET_RE.match(l):
            return True
        seen += 1
        if seen >= lookahead:
            break
    return False


In [20]:
from typing import Optional, Dict, Any, List, Union
from dataclasses import dataclass
from neo4j import GraphDatabase

# ----- DDL variants -----
CONSTRAINTS_V5 = [
    "CREATE CONSTRAINT policy_id IF NOT EXISTS FOR (p:Policy) REQUIRE p.id IS UNIQUE",
    "CREATE CONSTRAINT policy_section_id IF NOT EXISTS FOR (s:PolicySection) REQUIRE s.id IS UNIQUE",
    "CREATE CONSTRAINT policy_clause_id IF NOT EXISTS FOR (c:PolicyClause) REQUIRE c.id IS UNIQUE",
]
CONSTRAINTS_V4 = [
    "CREATE CONSTRAINT policy_id IF NOT EXISTS ON (p:Policy) ASSERT p.id IS UNIQUE",
    "CREATE CONSTRAINT policy_section_id IF NOT EXISTS ON (s:PolicySection) ASSERT s.id IS UNIQUE",
    "CREATE CONSTRAINT policy_clause_id IF NOT EXISTS ON (c:PolicyClause) ASSERT c.id IS UNIQUE",
]
FT_V5 = [
    "CREATE FULLTEXT INDEX ft_policy_clause IF NOT EXISTS "
    "FOR (c:PolicyClause) ON EACH [c.text, c.title, c.category]"
]
FT_V4_EXISTS = (
    "CALL db.indexes() YIELD name, type "
    "WHERE type='FULLTEXT' AND name='ft_policy_clause' "
    "RETURN count(*) AS cnt"
)
FT_V4_CREATE = (
    "CALL db.index.fulltext.createNodeIndex("
    "'ft_policy_clause', ['PolicyClause'], ['text','title','category'])"
)

# ----- Upsert & link queries (from your earlier version) -----
UPSERT_POLICY_NODE = """
MERGE (p:Policy {id:$policy.id})
  SET p.title=$policy.title,
      p.version=$policy.version,
      p.effective_on=date($policy.effective_on),
      p.owner=$policy.owner,
      p.uri=$policy.uri,
      p.sha256=$policy.sha256,
      p.status=coalesce(p.status,'active'),
      p.confidentiality=coalesce(p.confidentiality,'internal')
"""

UPSERT_SECTIONS_AND_CLAUSES = """
MATCH (p:Policy {id:$policy.id})
WITH p, $sections AS sections, $policy AS pol
UNWIND sections AS sec
  MERGE (s:PolicySection {id:sec.id})
    SET s.policy_id=pol.id,
        s.number=sec.number,
        s.title=sec.title,
        s.text=sec.text,
        s.sha256=sec.sha256
  MERGE (p)-[:HAS_SECTION]->(s)
  WITH s, sec, pol
  UNWIND sec.clauses AS cl
    MERGE (c:PolicyClause {id:cl.id})
      SET c.policy_id=pol.id,
          c.section_number=cl.section_number,
          c.number=cl.number,
          c.title=cl.title,
          c.text=cl.text,
          c.category=cl.category,
          c.path=cl.path,
          c.sha256=cl.sha256,
          c.is_heading=coalesce(cl.is_heading,false)
    MERGE (s)-[:HAS_CLAUSE]->(c)
"""



LINK_BY_TITLE_ONLY = """
UNWIND $links AS m
MATCH (cl:PolicyClause {id:m.clause_id})
UNWIND m.control_titles AS ttl
MATCH (sc:SubControl)
WHERE toLower(sc.title) = toLower(ttl)
MERGE (cl)-[r:ADDRESSES {source:coalesce(m.source,$default_source), at:date()}]->(sc);
"""

# Link by TITLE with framework scoping (recommended if you have multiple frameworks)
LINK_BY_TITLE_AND_FRAMEWORK = """
UNWIND $links AS m
MATCH (cl:PolicyClause {id:m.clause_id})
UNWIND m.control_titles AS ttl
MATCH (fw:Framework {name: m.framework_key})
MATCH (sc:SubControl)-[:IN_FRAMEWORK]->(fw)
WHERE toLower(sc.title) = toLower(ttl)
MERGE (cl)-[r:ADDRESSES {source:coalesce(m.source,$default_source), at:date()}]->(sc);
"""

LINK_RELATED_DOC_EXACT = """
MATCH (p:Policy {id:$policy_id})
MATCH (rd:RelatedDocument)
WHERE toLower(trim(rd.id)) = toLower(trim(p.title))
MERGE (rd)-[:REFERENCES {matched_on:'exact', at:date()}]->(p)
"""

# normalized match: remove spaces, hyphens, underscores; case-insensitive
LINK_RELATED_DOC_NORMALIZED = """
MATCH (p:Policy {id:$policy_id})
MATCH (rd:RelatedDocument)
WHERE NOT (rd)-[:REFERENCES]->(p)
WITH p, rd,
  toLower(replace(replace(replace(trim(rd.id),' ',''),'-',''),'_','')) AS rkey,
  toLower(replace(replace(replace(trim(p.title),' ',''),'-',''),'_',''))      AS pkey
WHERE rkey = pkey
MERGE (rd)-[:REFERENCES {matched_on:'normalized', at:date()}]->(p)
"""

In [21]:
# neo4j_loader.py (framework-agnostic)
from dataclasses import dataclass
from typing import Dict, List, Any, Optional, Union
from neo4j import GraphDatabase


@dataclass
class LoaderConfig:
    default_source: str = "auto-map"

class PolicyNeo4jLoader:
    def __init__(self, uri: str, user: str, password: str, database: str = "neo4j", config: Optional[LoaderConfig] = None):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.database = database
        self.cfg = config or LoaderConfig()

    def close(self):
        self.driver.close()

    def _neo4j_major(self) -> int:
        with self.driver.session(database=self.database) as s:
            ver = s.run(
                "CALL dbms.components() YIELD versions "
                "RETURN versions[0] AS version"
            ).single()["version"]
        return int(ver.split(".")[0])

    def ensure_schema(self):
        major = self._neo4j_major()
        with self.driver.session(database=self.database) as s:
            # Constraints
            stmts = CONSTRAINTS_V5 if major >= 5 else CONSTRAINTS_V4
            for cy in stmts:
                s.execute_write(lambda tx, q=cy: tx.run(q))

            # Full-text
            if major >= 5:
                for cy in FT_V5:
                    s.execute_write(lambda tx, q=cy: tx.run(q))
            else:
                cnt = s.run(FT_V4_EXISTS).single()["cnt"]
                if cnt == 0:
                    s.execute_write(lambda tx: tx.run(FT_V4_CREATE))

    def upsert_policy(self, policy_doc: dict):
        with self.driver.session(database=self.database) as s:
            # 1) Upsert the Policy node
            s.execute_write(lambda tx: tx.run(UPSERT_POLICY_NODE, policy=policy_doc))
            # 2) Upsert sections + clauses
            s.execute_write(lambda tx: tx.run(
                UPSERT_SECTIONS_AND_CLAUSES,
                policy=policy_doc,
                sections=policy_doc["sections"]
            ))
            # 3) Auto-link RelatedDoc -> Policy (exact, then normalized)
            s.execute_write(lambda tx: tx.run(
                LINK_RELATED_DOC_EXACT,
                policy_id=policy_doc["id"]
            ))
            s.execute_write(lambda tx: tx.run(
                LINK_RELATED_DOC_NORMALIZED,
                policy_id=policy_doc["id"]
            ))

    def link_controls(self, links: List[Dict[str, Any]], *, framework_key: Optional[str] = None):
        if not links:
            return
        with self.driver.session(database=self.database) as s:
            # Title-based linking takes precedence if present
            if "control_titles" in (links[0] or {}):
                cypher = (LINK_BY_TITLE_AND_FRAMEWORK if framework_key else LINK_BY_TITLE_ONLY)
                s.execute_write(lambda tx: tx.run(
                    cypher,
                    links=links,
                    default_source=self.cfg.default_source
                ))
                return

            # Fallback: ID-based linking (existing behavior)
            if framework_key:
                s.execute_write(lambda tx: tx.run(
                    LINK_BY_ID_AND_FRAMEWORK,
                    links=links,
                    default_source=self.cfg.default_source
                ))
            else:
                s.execute_write(lambda tx: tx.run(
                    LINK_BY_ID_ONLY,
                    links=links,
                    default_source=self.cfg.default_source
                ))


In [22]:
import re
from bs4 import BeautifulSoup

CONTROL_CODE_RE = re.compile(r"\((CC\d\.\d|CC\d|A\d\.\d)\)|\b(CC\d\.\d|A\d\.\d)\b", re.I)

def extract_control_id(title: str) -> str:
    """Return canonical control ID (e.g., 'CC6.6') from a title string."""
    if not title:
        return ""
    m = CONTROL_CODE_RE.search(title)
    if m:
        return m.group(1) or m.group(2)
    return re.sub(r"\W+", "", title)[:20].upper()  # fallback

In [23]:
# soc2_autolinker.py
import re, json, html
from pathlib import Path
from typing import Dict, List, Any, Tuple
from collections import defaultdict

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

CONTROL_CODE_RE = re.compile(r"\((CC\d\.\d|CC\d|A\d\.\d)\)|\b(CC\d\.\d|A\d\.\d)\b", re.I)

def clean_html(s: str) -> str:
    s = html.unescape(s or "")
    s = re.sub(r"<br\s*/?>", "\n", s, flags=re.I)
    s = re.sub(r"<[^>]+>", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def control_id_from_title(title: str) -> str:
    m = CONTROL_CODE_RE.search(title or "")
    return (m.group(1) if m else re.sub(r"\W+", "", title or "")).upper()

import json, re
from pathlib import Path
from typing import Any, Dict, List, Union

CONTROL_CODE_RE = re.compile(r"\((CC\d\.\d|CC\d|A\d\.\d)\)|\b(CC\d\.\d|A\d\.\d)\b", re.I)

def extract_control_id(title: str) -> str:
    m = CONTROL_CODE_RE.search(title or "")
    return (m.group(1) or m.group(2)) if m else re.sub(r"\W+", "", title or "")[:20].upper()

def _as_obj(data_or_path: Union[str, Dict[str, Any]]) -> Dict[str, Any]:
    # Path to JSON file?
    if isinstance(data_or_path, (str, Path)) and Path(str(data_or_path)).exists():
        return json.loads(Path(str(data_or_path)).read_text(encoding="utf-8"))
    # Raw JSON string?
    if isinstance(data_or_path, str):
        return json.loads(data_or_path)
    # Already a dict?
    if isinstance(data_or_path, dict):
        return data_or_path
    raise TypeError(f"Unsupported type: {type(data_or_path)}")

def _get_sub_controls(obj: Any) -> List[Dict[str, Any]]:
    # Direct list?
    if isinstance(obj, list) and obj and isinstance(obj[0], dict) and "title" in obj[0]:
        return obj
    # Common shapes
    if isinstance(obj, dict):
        if "sub_controls" in obj and isinstance(obj["sub_controls"], list):
            return obj["sub_controls"]
        if "data" in obj and isinstance(obj["data"], dict) and "sub_controls" in obj["data"]:
            return obj["data"]["sub_controls"]
    # Fallback: search for the first list of dicts containing "title"
    if isinstance(obj, dict):
        for v in obj.values():
            sc = _get_sub_controls(v)
            if sc:
                return sc
    raise KeyError("Could not locate 'sub_controls' in the provided JSON.")

def convert_html_content_to_list(html_content):
    
    soup = BeautifulSoup(html_content, "html.parser")
    
    # Prepare the base list
    structured_content = []

    # Extract sections by `sub-sub-head` class
    sections = soup.find_all("div", class_="sub-sub-head")
    for section in sections:
        section_title = section.get_text(strip=True).replace(":", "")  # Extract section title
        section_content = section.next_sibling

        content = {"title": section_title, "items": []}

        # Collect all subsequent sibling elements until the next section
        sibling = section
        while sibling and (sibling := sibling.next_sibling):
            if sibling.name == "div" and "sub-sub-head" in sibling.get("class", []):
                break  # Stop at the next section
            if sibling.string and sibling.string.strip():
                # Handle numbered items
                for line in sibling.string.split("<br/>"):
                    line = line.strip()
                    if line and line[0].isdigit():  # Remove leading numbers
                        item_text = line.split(")", 1)[1].strip() if ")" in line else line
                        content["items"].append(item_text)
                    else:
                        content["items"].append(line)

        try:
            if not content["items"] and section_content:
                text = str(section_content).strip()
                if text:
                    content["items"].append(text)

        except Exception as e:
            print(f"Error: {e}")
            print(f"Section: {section_title}")
            print(f"Content: {section_content}")
            print(f"Items: {content['items']}")

        structured_content.append(content)

    return structured_content

def load_controls(data_or_path: Union[str, Dict[str, Any]]) -> List[Dict[str, Any]]:
    obj = _as_obj(data_or_path)
    sub_controls = _get_sub_controls(obj)

    controls = []
    for sc in sub_controls:
        title = sc.get("title", "")
        desc  = sc.get("description", "") or ""
        cid   = extract_control_id(title)
        # Parse related docs from the HTML-ish description using your helper
        try:
            related = convert_html_content_to_list(desc)
        except Exception:
            related = []
        controls.append({
            "id": cid,                # e.g., "CC6.6"
            "title": title,           # e.g., "Protecting Against External Threats (CC6.6)"
            "description": desc,
            "related_docs": related,  # [{title:..., items:[...]}] per your helper
            "text": f"{title} {desc}".strip() 
        })
    return controls


import re

def build_related_doc_lexicon(controls):
    """
    Map control_id -> list of doc-hint strings.
    Works whether related_docs is:
      - a string (HTML/text)
      - a list of sections like [{'title': 'Related Documents', 'items': [...]}]
    """
    def _norm(s: str) -> str:
        return re.sub(r"\s+", " ", (s or "").strip()).lower()

    lex = {}
    for c in controls:
        cid = c["id"]
        hints = []

        rd = c.get("related_docs")

        # related_docs as LIST of sections from convert_html_content_to_list(...)
        if isinstance(rd, list):
            for sec in rd:
                # include the section title itself as a weak hint (optional)
                sec_title = _norm(sec.get("title", ""))
                if sec_title and len(sec_title) >= 5:
                    hints.append(sec_title)

                for item in sec.get("items", []) or []:
                    t = _norm(item)
                    if len(t) >= 5:
                        # strip leading list markers "1) ", "-", "•", etc.
                        t = t.lstrip(" -•*0123456789).(").strip()
                        if len(t) >= 5:
                            hints.append(t)

        # related_docs as STRING (fallback)
        elif isinstance(rd, str):
            for line in rd.split("\n"):
                t = _norm(line).lstrip(" -•*0123456789).(").strip()
                if len(t) >= 5:
                    hints.append(t)

        # dedupe while preserving order
        seen = set()
        uniq = []
        for h in hints:
            if h not in seen:
                uniq.append(h)
                seen.add(h)

        lex[cid] = uniq

    return lex


def text_for_clause(cl: Dict[str, Any]) -> str:
    parts = [cl.get("title") or "", cl.get("text") or "", cl.get("path") or ""]
    return " ".join(p for p in parts if p).strip()

def infer_map(
    policy_doc: Dict[str, Any],
    soc2_controls: List[Dict[str,Any]],
    *,
    top_k: int = 2,
    sim_threshold: float = 0.18,
    hint_boost: float = 0.10,
    rule_boost: float = 0.08,
    embeddings_fn = None  # optional: fn(list_of_texts)->ndarray
) -> Dict[str, List[str]]:
    """
    Returns a dict: { <clause.number> : [control_id, ...] }
    """
    # 1) Gather clause texts
    clauses = []
    for sec in policy_doc["sections"]:
        for cl in sec["clauses"]:
            if cl.get("is_heading"):
                continue
            clauses.append(cl)

    clause_texts = [text_for_clause(c) for c in clauses]
    control_texts = [c["text"] for c in soc2_controls]

    # 2) Base similarity (TF-IDF or embeddings)
    if embeddings_fn:
        import numpy as np
        emb_clauses = embeddings_fn(clause_texts)
        emb_controls = embeddings_fn(control_texts)
        base_sim = cosine_similarity(emb_clauses, emb_controls)
    else:
        vec = TfidfVectorizer(stop_words="english", ngram_range=(1,2), min_df=1)
        X = vec.fit_transform(clause_texts + control_texts)
        nC = len(clause_texts)
        base_sim = cosine_similarity(X[:nC], X[nC:])

    # 3) Add boosts from related-doc hints and lightweight rules
    hints = build_related_doc_lexicon(soc2_controls)

    def norm(s: str) -> str:
        return re.sub(r"\s+", " ", s or "").lower()

    for i, cl in enumerate(clauses):
        cl_txt = norm(text_for_clause(cl))
        cl_cat = (cl.get("category") or "").lower()

        for j, ctrl in enumerate(soc2_controls):
            # hint boost: if any related-doc hint string appears in the clause text, boost
            for h in hints.get(ctrl["id"], []):
                if h and h in cl_txt:
                    base_sim[i, j] += hint_boost
                    break

            # rule boosts (generic)
            cid = ctrl["id"]
            if any(k in cl_cat or k in cl_txt for k in ["internet","email","acceptable use","mobile","software","security"]):
                if cid.startswith("CC6."):           # Access/security family
                    base_sim[i, j] += rule_boost
            if any(k in cl_cat or k in cl_txt for k in ["logging","monitor","siem","ids","anomalies"]):
                if cid.startswith("CC7.1") or cid.startswith("CC7.2"):
                    base_sim[i, j] += rule_boost
            if any(k in cl_cat or k in cl_txt for k in ["incident","breach","escalation","response","recovery"]):
                if cid in ("CC7.3","CC7.4","CC7.5"):
                    base_sim[i, j] += rule_boost
            if any(k in cl_cat or k in cl_txt for k in ["change management","change", "deployment", "version control","rollback"]):
                if cid == "CC8.1":
                    base_sim[i, j] += rule_boost
            if any(k in cl_cat or k in cl_txt for k in ["vendor","third party","nondisclosure","nda","supplier"]):
                if cid == "CC9.2":
                    base_sim[i, j] += rule_boost
            if any(k in cl_cat or k in cl_txt for k in ["business continuity","disaster recovery","bcp","dr"]):
                if cid in ("CC9.1","A1.2","A1.3"):
                    base_sim[i, j] += rule_boost
            if any(k in cl_cat or k in cl_txt for k in ["capacity","availability","threshold","scaling"]):
                if cid == "A1.1":
                    base_sim[i, j] += rule_boost

    # 4) Select top-k above threshold
    mapping: Dict[str, List[str]] = defaultdict(list)
    for i, cl in enumerate(clauses):
        sims = list(enumerate(base_sim[i]))
        sims.sort(key=lambda t: t[1], reverse=True)
        chosen = [soc2_controls[j]["id"] for j, s in sims[:top_k] if s >= sim_threshold]
        if chosen:
            mapping[cl["number"]] = chosen

    return mapping

# -------- convenience to build links for Neo4j
def build_links_for_loader(policy_doc: Dict[str,Any], number_to_controls: Dict[str, List[str]], source="soc2-auto"):
    idx = {cl["number"]: cl["id"]
           for sec in policy_doc["sections"] for cl in sec["clauses"] if not cl.get("is_heading")}
    links = []
    for num, ctrls in number_to_controls.items():
        if num in idx:
            links.append({"clause_id": idx[num], "control_ids": ctrls, "source": source})
    return links


In [24]:
import re
from typing import Dict, Any, List, Optional, Union

_CODE_RE = re.compile(r"(CC\d\.\d|CC\d|A\d\.\d)", re.I)

def _build_control_lookups(controls: List[Dict[str, Any]]):
    # Normalize keys for flexible matching
    def norm(s: str) -> str:
        return (s or "").strip().lower()
    by_id = {c["id"]: c for c in controls if c.get("id")}
    by_title = {norm(c.get("title","")): c for c in controls if c.get("title")}
    return by_id, by_title, norm

def _resolve_label_to_id(label: str, by_id: Dict[str,Any], by_title: Dict[str,Any], norm) -> Optional[str]:
    """
    Accepts either an ID ('CC6.1') or a title ('Protecting Information Assets (CC6.1)').
    Returns canonical ID or None if not found.
    """
    if not label:
        return None
    # 1) direct ID match
    if label in by_id:
        return label
    # 2) exact title match (case-insensitive)
    t = by_title.get(norm(label))
    if t:
        return t["id"]
    # 3) extract code from string and match by id
    m = _CODE_RE.search(label)
    if m and m.group(1) in by_id:
        return m.group(1)
    return None

from typing import Dict, Any, List, Optional, Union

def build_links_from_map(
    policy_doc: dict,
    mapping: Dict[str, Union[List[str], List[Dict[str,str]]]],
    *,
    source: str = "auto-map",
    framework_key: Optional[str] = None,
    link_by: str = "title"  # 'title' or 'id'
) -> List[Dict[str, Any]]:
    """
    Build link payloads from a clause-number -> controls map.
    If link_by='title', mapping values are treated as SubControl titles.
    If link_by='id', mapping values are treated as SubControl ids.
    """
    # index clause numbers -> clause ids
    idx = {
        cl["number"]: cl["id"]
        for sec in policy_doc["sections"]
        for cl in sec["clauses"]
        if not cl.get("is_heading")
    }

    rows: List[Dict[str, Any]] = []
    for clause_number, ctrl_spec in mapping.items():
        clause_id = idx.get(clause_number)
        if not clause_id:
            continue

        row: Dict[str, Any] = {"clause_id": clause_id, "source": source}

        # Namespaced controls (object shape) are treated as IDs; keep for compatibility
        if ctrl_spec and isinstance(ctrl_spec[0], dict):
            row["controls"] = ctrl_spec
            row["control_ids"] = []
        else:
            if link_by == "title":
                row["control_titles"] = list(ctrl_spec)  # pass titles through
            else:
                row["control_ids"] = list(ctrl_spec)     # legacy path
            if framework_key:
                row["framework_key"] = framework_key

        rows.append(row)

    return rows


In [25]:
controls = load_controls("data/frameworks/soc_2.json")
cfg = LoaderConfig()
cfg.control_label = "SubControl" 

In [26]:
def process_text(file_location, policy_id, title, version, owner, policy_uri, controls, db_name) -> None: 
    try:
        with open(file_location, "r", encoding="utf-8") as file:
            text = file.read()
    except UnicodeDecodeError:
        with open(file_location, "r", encoding="latin-1") as file:
            text = file.read()

    
    text = text.replace("\r\n", "\n")  # Normalize line endings


    policy = parse_policy(
        text=text,
        policy_id=policy_id,
        title=title,
        version=version,
        owner=owner,
        policy_uri=policy_uri
    )
    soc2_map = infer_map(policy, controls, top_k=2, sim_threshold=0.18)
    control_lookup = {c["id"]: c.get("title", c["id"]) for c in controls}
    map_titles = {
        clause_num: [control_lookup.get(cid, cid) for cid in ctrl_ids]
        for clause_num, ctrl_ids in soc2_map.items()
    }


    loader = PolicyNeo4jLoader(
        uri=NEO4J_URI,
        user=NEO4J_USER,
        password=NEO4J_PASSWORD,
        database=db_name,
        config=cfg
    )

    loader.ensure_schema()
    loader.upsert_policy(policy)
    links = build_links_from_map(policy, map_titles)
    loader.link_controls(links)
    loader.close()

In [ ]:
#################################  IMPORTANT  ######################################
#need to re-write this with all policies in the polocies folder

with open("all_policies.json", "r") as f:
    all_policies = json.load(f)
for policy in all_policies:
    process_text(
        file_location=policy.get('file_location'),
        policy_id=policy.get('policy_id'),
        title=policy.get('title'),
        version=policy.get('version'),
        owner=policy.get('owner'),
        policy_uri=policy.get('policy_uri'),
        controls=controls,
        db_name=policy.get('db_name')
    )

In [28]:
file_location = "policies/SOC2/Logging_and_Monitoring_Policy.txt"
policy_id = "SOC2-LMP-v1"
title = "Logging and Monitoring Policy"
version = "1.0"
owner = "Chief Technology Officer"
policy_uri = "blob://policies/SOC2-LMP-v1.pdf"
db_name = "soc-2"

try:
    with open(file_location, "r", encoding="utf-8") as file:
        text = file.read()
except UnicodeDecodeError:
    with open(file_location, "r", encoding="latin-1") as file:
        text = file.read()
            
policy = parse_policy(
        text=text,
        policy_id=policy_id,
        title=title,
        version=version,
        owner=owner,
        policy_uri=policy_uri
    )

In [ ]:
policy

{'id': 'SOC2-LMP-v1',
 'title': 'Logging and Monitoring Policy',
 'version': '1.0',
 'effective_on': '2025-10-28',
 'owner': 'Chief Technology Officer',
 'uri': 'blob://policies/SOC2-LMP-v1.pdf',
 'sha256': 'b9f4a486e24665f8e328e64ee28d60eb40a7d0dc01f3fac71f608fa2b704bd39',
 'sections': [{'id': 'SOC2-LMP-v1:S1',
   'number': '1',
   'title': 'PURPOSE',
   'text': '<<COMPANY>> has developed and implemented this policy to address internal controls, industry best practices. The purpose of this policy is to document and communicate the control requirements contained within the <<COMPANY>> Security Program. Logging and monitoring components are required to effectively assess information system controls, operations, and general security. This policy provides a set of logging policies and procedures aimed to establish baseline components across the <<COMPANY>> environment.',
   'sha256': '4b79d86d4474e6616f9301eae42b6e8d6257763eac8db68a4ac314330c32038b',
   'clauses': [{'id': 'SOC2-LMP-v1:1.c

: 